# 02_eda_model_suitability

Run automated checks that suggest suitability for Regression, Classification, Clustering, Anomaly Detection, Neural Networks, and Learning Type.

In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

base = Path('..') / 'data'
files = [p for p in base.iterdir() if p.is_file()]

def analyze(df):
    out = {}
    n = len(df)
    out['rows'] = n
    out['num_features'] = df.select_dtypes(include='number').shape[1]
    out['cat_features'] = df.select_dtypes(include=['object','category']).shape[1]
    # simple target heuristic: columns named like 'target','class','label','y' or common suffix
    candidates = [c for c in df.columns if any(k in c.lower() for k in ('target','class','label','y','flag','churn','fraud','claim','score'))]
    out['target_candidates'] = candidates
    # imbalance check for candidate targets
    if candidates:
        for t in candidates[:3]:
            vc = df[t].value_counts(dropna=False)
            out[f'imbl_{t}'] = vc.to_dict()
    # clustering feasibility
    num = df.select_dtypes(include='number').dropna(axis=1, how='all')
    if num.shape[1] >= 2 and len(num) >= 10:
        sample = num.sample(n=min(2000, len(num)), random_state=1)
        try:
            kmeans = KMeans(n_clusters=3, random_state=1).fit(sample)
            out['clustering_silhouette'] = float(silhouette_score(sample, kmeans.labels_))
        except Exception as e:
            out['clustering_error'] = str(e)
    # anomaly estimation
    if num.shape[1] >= 1 and len(num) >= 50:
        iso = IsolationForest(contamination=0.01, random_state=1)
        try:
            preds = iso.fit_predict(num.sample(n=min(2000, len(num)), random_state=1))
            out['anomaly_rate_est'] = float((preds==-1).mean())
        except Exception as e:
            out['anomaly_error'] = str(e)
    return out

for p in files:
    try:
        suf = p.suffix.lower()
        if suf in {'.csv','.txt'}:
            df = pd.read_csv(p)
        elif suf in {'.xlsx','.xls'}:
            df = pd.read_excel(p)
        elif suf in {'.parquet'}:
            df = pd.read_parquet(p)
        else:
            print('skip', p.name)
            continue
    except Exception as e:
        print('failed to load', p.name, e)
        continue
    print('\n---', p.name, '---')
    res = analyze(df)
    for k,v in res.items():
        print(k, ':', v)
    # short recommendation heuristic
    recs = []
    if res['num_features'] >= 2 and res['rows'] > 100:
        recs.append('Regression: possible (if continuous target present)')
        recs.append('Classification: possible (if categorical target present)')
    if res.get('clustering_silhouette'):
        if res['clustering_silhouette'] > 0.1:
            recs.append('Clustering: promising (silhouette > 0.1)')
        else:
            recs.append('Clustering: weak (low silhouette)')
    if res.get('anomaly_rate_est', 0) > 0:
        recs.append('Anomaly Detection: feasible (IsolationForest sample)')
    if res['rows'] >= 10000 or res['num_features'] >= 50:
        recs.append('Neural Networks: feasible at scale (consider embeddings for categorical features)')
    if not recs:
        recs.append('Small dataset / limited numeric features — careful feature engineering or data augmentation recommended')
    print('\nRecommendations:')
    for r in recs:
        print('-', r)
